In [1]:
from datasets import load_dataset
dataset = load_dataset("ButterChicken98/plantvillage-image-text-pairs")
dataset


DatasetDict({
    train: Dataset({
        features: ['image', 'caption', 'captions'],
        num_rows: 20638
    })
})

In [2]:
import pandas as pd
df = dataset["train"].to_pandas()
df.head()

,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."


In [3]:
df.to_csv("dataset_raw.csv", index=False)
print("CSV created!")

CSV created!


In [4]:
#expand caption list into multiple rows
df = df.explode("captions", ignore_index=True)
#rename captions column->text
df = df.rename(columns={"captions": "text"})
df.head()

,image,caption,text
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,A vibrant green and healthy tomato leaf with s...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,"A healthy Solanum lycopersicum leaf, free of d..."
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,"A fresh tomato leaf outdoors, glowing in sunli..."
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,"A clean and healthy tomato leaf image, perfect..."
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,A tomato leaf showing dark brown lesions and w...


In [5]:
def extract_label(text):
  text = text.lower()

  if "healthy" in text:
    return "healthy"
  elif "blight" in text:
    return "blight"
  elif "mildew" in text:
    return "mildew"
  elif "rust" in text:
    return "rust"
  elif "spot" in text:
    return "leaf_spot"
  else:
    return "other"
df["label_name"] = df["text"].apply(extract_label)

In [6]:
import pandas as pd

# Load the raw dataset which contains the original 'caption' column
df = pd.read_csv("dataset_raw.csv")
print("DataFrame loaded from dataset_raw.csv:")
display(df.head())

DataFrame loaded from dataset_raw.csv:


,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,['A vibrant green and healthy tomato leaf with...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,['A tomato leaf showing dark brown lesions and...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,['A vibrant green and healthy tomato leaf with...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,['A tomato leaf with mosaic-like patterns of l...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,['A fresh green bell pepper leaf with a smooth...


In [7]:
# Explode the 'captions' list into multiple rows and rename it to 'text'
df = df.explode("captions", ignore_index=True)
df = df.rename(columns={"captions": "text"})

# Rename the original 'caption' column to 'label_name' to be used as the classification label
df = df.rename(columns={"caption": "label_name"})
print("DataFrame after exploding captions and renaming columns:")
display(df.head())

DataFrame after exploding captions and renaming columns:


,image,label_name,text
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,['A vibrant green and healthy tomato leaf with...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,['A tomato leaf showing dark brown lesions and...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,['A vibrant green and healthy tomato leaf with...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,['A tomato leaf with mosaic-like patterns of l...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,['A fresh green bell pepper leaf with a smooth...


In [8]:
# Select only the 'label_name' and 'text' columns for the final dataframe
df = df[["label_name", "text"]]
print("Final DataFrame for classification:")
display(df.head())

Final DataFrame for classification:


,label_name,text
0,Tomato healthy,['A vibrant green and healthy tomato leaf with...
1,Tomato Late blight,['A tomato leaf showing dark brown lesions and...
2,Tomato healthy,['A vibrant green and healthy tomato leaf with...
3,Tomato mosaic virus,['A tomato leaf with mosaic-like patterns of l...
4,Pepper bell healthy,['A fresh green bell pepper leaf with a smooth...


In [9]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["label_name"])
num_classes = len(encoder.classes_)

print("Number of classes using original captions:", num_classes)
print("Classes:", encoder.classes_)

Number of classes using original captions: 15
Classes: ['Pepper bell Bacterial spot' 'Pepper bell healthy' 'Potato Early blight'
 'Potato Late blight' 'Potato healthy' 'Tomato Bacterial spot'
 'Tomato Early blight' 'Tomato Late blight' 'Tomato Leaf Mold'
 'Tomato Septoria leaf spot' 'Tomato Spider mites Two spotted spider mite'
 'Tomato Target Spot' 'Tomato YellowLeaf Curl Virus' 'Tomato healthy'
 'Tomato mosaic virus']


In [10]:
df = df[["label_name", "text"]]
df.head()

,label_name,text
0,Tomato healthy,['A vibrant green and healthy tomato leaf with...
1,Tomato Late blight,['A tomato leaf showing dark brown lesions and...
2,Tomato healthy,['A vibrant green and healthy tomato leaf with...
3,Tomato mosaic virus,['A tomato leaf with mosaic-like patterns of l...
4,Pepper bell healthy,['A fresh green bell pepper leaf with a smooth...


In [11]:
df.to_csv("dataset.csv", index=False)
print("CSV created successfully")

CSV created successfully


In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

!pip install evaluate
import evaluate

In [13]:
df = pd.read_csv("dataset.csv")
df.head()

,label_name,text
0,Tomato healthy,['A vibrant green and healthy tomato leaf with...
1,Tomato Late blight,['A tomato leaf showing dark brown lesions and...
2,Tomato healthy,['A vibrant green and healthy tomato leaf with...
3,Tomato mosaic virus,['A tomato leaf with mosaic-like patterns of l...
4,Pepper bell healthy,['A fresh green bell pepper leaf with a smooth...


In [14]:
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["label_name"])
num_classes = len(encoder.classes_)
print("Number of classes:", num_classes)
print("Classes:", encoder.classes_)



Number of classes: 15
Classes: ['Pepper bell Bacterial spot' 'Pepper bell healthy' 'Potato Early blight'
 'Potato Late blight' 'Potato healthy' 'Tomato Bacterial spot'
 'Tomato Early blight' 'Tomato Late blight' 'Tomato Leaf Mold'
 'Tomato Septoria leaf spot' 'Tomato Spider mites Two spotted spider mite'
 'Tomato Target Spot' 'Tomato YellowLeaf Curl Virus' 'Tomato healthy'
 'Tomato mosaic virus']


In [15]:
df_train,df_test = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(df_train,preserve_index=False)
test_dataset = Dataset.from_pandas(df_test,preserve_index=False)

print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))


Train dataset size: 16510
Test dataset size: 4128


In [16]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [17]:
def tokenize(batch):
  return tokenizer(batch["text"], truncation=True,max_length=128)
tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_test = test_dataset.map(tokenize, batched=True)



Map:   0%|          | 0/16510 [00:00<?, ? examples/s]

Map:   0%|          | 0/4128 [00:00<?, ? examples/s]

In [18]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes
    )

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
  logits,labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  return accuracy.compute(predictions=predictions, references=labels)

In [20]:
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)


In [21]:

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [22]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [23]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_test = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/16510 [00:00<?, ? examples/s]

Map:   0%|          | 0/4128 [00:00<?, ? examples/s]

In [24]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [26]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")


CUDA available: True


In [27]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.001009,0.000106,1.000000
2,0.000112,0.000034,1.000000
3,0.000047,0.000018,1.000000
4,0.000028,0.000011,1.000000
5,0.000020,0.000010,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=5160, training_loss=0.00024320667883915493, metrics={'train_runtime': 1010.6759, 'train_samples_per_second': 81.678, 'train_steps_per_second': 5.105, 'total_flos': 2458303673495040.0, 'train_loss': 0.00024320667883915493, 'epoch': 5.0})

In [28]:
eval_results = trainer.evaluate(tokenized_test)
print(eval_results)

{'eval_loss': 9.512869837635662e-06, 'eval_accuracy': 1.0, 'eval_runtime': 8.7685, 'eval_samples_per_second': 470.777, 'eval_steps_per_second': 29.424, 'epoch': 5.0}


In [29]:
predictions_output = trainer.predict(tokenized_test)
predictions = np.argmax(predictions_output.predictions, axis=1)
print(f"Shape of predictions: {predictions.shape}")

Shape of predictions: (4128,)


In [30]:
print("Evaluation Results:", eval_results)

Evaluation Results: {'eval_loss': 9.512869837635662e-06, 'eval_accuracy': 1.0, 'eval_runtime': 8.7685, 'eval_samples_per_second': 470.777, 'eval_steps_per_second': 29.424, 'epoch': 5.0}


In [31]:
sample_df = df_test.copy()
sample_df["predicted_label"] = predictions
sample_df["predicted_label_name"] = encoder.inverse_transform(predictions)
print("Sample of actual vs. predicted labels:")
print(sample_df[['text', 'label_name', 'label', 'predicted_label_name', 'predicted_label']].head())

Sample of actual vs. predicted labels:
                                                    text  \
12412  ['A tomato leaf with small, water-soaked spots...   
3024   ['A tomato leaf with dark brown spots and conc...   
9547   ['A potato leaf with concentric brown rings fo...   
16532  ['A tomato leaf with dark brown spots and conc...   
12591  ['A fresh green bell pepper leaf with a smooth...   

                  label_name  label   predicted_label_name  predicted_label  
12412  Tomato Bacterial spot      5  Tomato Bacterial spot                5  
3024     Tomato Early blight      6    Tomato Early blight                6  
9547     Potato Early blight      2    Potato Early blight                2  
16532    Tomato Early blight      6    Tomato Early blight                6  
12591    Pepper bell healthy      1    Pepper bell healthy                1  


EXTRACT LABELS-15 classes

In [32]:
print(f"Number of classes: {num_classes}")
print(f"Class names: {encoder.classes_}")

Number of classes: 15
Class names: ['Pepper bell Bacterial spot' 'Pepper bell healthy' 'Potato Early blight'
 'Potato Late blight' 'Potato healthy' 'Tomato Bacterial spot'
 'Tomato Early blight' 'Tomato Late blight' 'Tomato Leaf Mold'
 'Tomato Septoria leaf spot' 'Tomato Spider mites Two spotted spider mite'
 'Tomato Target Spot' 'Tomato YellowLeaf Curl Virus' 'Tomato healthy'
 'Tomato mosaic virus']


## Save Updated DataFrame



In [33]:
df.to_csv("dataset.csv", index=False)
print("DataFrame with 15 classes saved to dataset.csv!")

DataFrame with 15 classes saved to dataset.csv!


## Prepare Data for Training



In [34]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

# 1. Load the updated 'dataset.csv' file
df = pd.read_csv("dataset.csv")

# 2. Initialize a LabelEncoder object and 3. Fit and transform 'label_name' to 'label'
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["label_name"])
num_classes = len(encoder.classes_)

print(f"Number of classes after re-encoding: {num_classes}")
print(f"Class names after re-encoding: {encoder.classes_}")

# 4. Split the df DataFrame into training and testing sets
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# 5. Convert df_train and df_test into Dataset objects
train_dataset = Dataset.from_pandas(df_train, preserve_index=False)
test_dataset = Dataset.from_pandas(df_test, preserve_index=False)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Ensure tokenizer is loaded (it might have been reset or not globally available in this specific context)
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 6. Apply the existing tokenize function to both train_dataset and test_dataset
def tokenize(batch):
  return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_test = test_dataset.map(tokenize, batched=True)

print("Tokenization complete for train and test datasets.")

Number of classes after re-encoding: 15
Class names after re-encoding: ['Pepper bell Bacterial spot' 'Pepper bell healthy' 'Potato Early blight'
 'Potato Late blight' 'Potato healthy' 'Tomato Bacterial spot'
 'Tomato Early blight' 'Tomato Late blight' 'Tomato Leaf Mold'
 'Tomato Septoria leaf spot' 'Tomato Spider mites Two spotted spider mite'
 'Tomato Target Spot' 'Tomato YellowLeaf Curl Virus' 'Tomato healthy'
 'Tomato mosaic virus']
Train dataset size: 16510
Test dataset size: 4128


Map:   0%|          | 0/16510 [00:00<?, ? examples/s]

Map:   0%|          | 0/4128 [00:00<?, ? examples/s]

Tokenization complete for train and test datasets.


## Initialize Model




In [35]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes
)

print("Model re-initialized with num_labels:", num_classes)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model re-initialized with num_labels: 15


## Train Model

### Subtask:
Re-train the model using the updated datasets and model configuration.


**Reasoning**:
The subtask is to re-train the model. I need to call the `.train()` method on the `trainer` object to start the training process.



In [43]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.061885,0.000128,1.000000
2,0.000127,0.000036,1.000000
3,0.000050,0.000017,1.000000
4,0.000028,0.000011,1.000000
5,0.000021,0.000009,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=5160, training_loss=0.012422203177282976, metrics={'train_runtime': 1057.8734, 'train_samples_per_second': 78.034, 'train_steps_per_second': 4.878, 'total_flos': 2392621876306080.0, 'train_loss': 0.012422203177282976, 'epoch': 5.0})

In [44]:
eval_results = trainer.evaluate(tokenized_test)
print(eval_results)

{'eval_loss': 9.185630915453658e-06, 'eval_accuracy': 1.0, 'eval_runtime': 14.9457, 'eval_samples_per_second': 276.201, 'eval_steps_per_second': 17.263, 'epoch': 5.0}


## Generate Predictions



In [45]:
predictions_output = trainer.predict(tokenized_test)
predictions = np.argmax(predictions_output.predictions, axis=1)
print(f"Shape of predictions: {predictions.shape}")

Shape of predictions: (4128,)


In [46]:
sample_df = df_test.copy()
sample_df["predicted_label"] = predictions
sample_df["predicted_label_name"] = encoder.inverse_transform(predictions)
print("Sample of actual vs. predicted labels:")
print(sample_df[['text', 'label_name', 'label', 'predicted_label_name', 'predicted_label']].head())

Sample of actual vs. predicted labels:
                                                    text  \
12412  ['A tomato leaf with small, water-soaked spots...   
3024   ['A tomato leaf with dark brown spots and conc...   
9547   ['A potato leaf with concentric brown rings fo...   
16532  ['A tomato leaf with dark brown spots and conc...   
12591  ['A fresh green bell pepper leaf with a smooth...   

                  label_name  label   predicted_label_name  predicted_label  
12412  Tomato Bacterial spot      5  Tomato Bacterial spot                5  
3024     Tomato Early blight      6    Tomato Early blight                6  
9547     Potato Early blight      2    Potato Early blight                2  
16532    Tomato Early blight      6    Tomato Early blight                6  
12591    Pepper bell healthy      1    Pepper bell healthy                1  


In [47]:
print("Evaluation Results:", eval_results)

Evaluation Results: {'eval_loss': 9.185630915453658e-06, 'eval_accuracy': 1.0, 'eval_runtime': 14.9457, 'eval_samples_per_second': 276.201, 'eval_steps_per_second': 17.263, 'epoch': 5.0}


## Save Model



In [48]:
output_dir = "./fine_tuned_model"
trainer.save_model(output_dir)
print(f"Model saved to {output_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./fine_tuned_model


## Load Model



In [49]:
from transformers import AutoModelForSequenceClassification

loaded_model = AutoModelForSequenceClassification.from_pretrained(output_dir)
print("Model loaded successfully!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully!


## Prepare for Inference



In [50]:
def preprocess_text(text):
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        return_tensors="pt"
    )
    return inputs

print("preprocess_text function defined.")

preprocess_text function defined.


## Make Predictions



In [51]:
import torch

def predict_disease_label(text):
    # Preprocess the text
    inputs = preprocess_text(text)

    # Make prediction
    with torch.no_grad():
        outputs = loaded_model(**inputs)

    # Get predicted class (logits)
    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()

    # Convert class ID to human-readable label
    predicted_label = encoder.inverse_transform([predicted_class_id])[0]

    return predicted_label

# Example usage:
sample_text_1 = "A tomato leaf with circular brown spots and target-like lesions."
sample_text_2 = "This is a healthy pepper bell plant with no signs of disease."
sample_text_3 = "A potato leaf showing signs of early blight."

prediction_1 = predict_disease_label(sample_text_1)
prediction_2 = predict_disease_label(sample_text_2)
prediction_3 = predict_disease_label(sample_text_3)

print(f"Sample text 1: '{sample_text_1}'\nPredicted label: {prediction_1}\n")
print(f"Sample text 2: '{sample_text_2}'\nPredicted label: {prediction_2}\n")
print(f"Sample text 3: '{sample_text_3}'\nPredicted label: {prediction_3}\n")

Sample text 1: 'A tomato leaf with circular brown spots and target-like lesions.'
Predicted label: Tomato Septoria leaf spot

Sample text 2: 'This is a healthy pepper bell plant with no signs of disease.'
Predicted label: Pepper bell healthy

Sample text 3: 'A potato leaf showing signs of early blight.'
Predicted label: Potato Late blight



In [52]:
new_sample_text = "A tomato leaf with early blight symptoms, including dark spots and yellowing."
new_prediction = predict_disease_label(new_sample_text)

print(f"New Sample Text: '{new_sample_text}'\nPredicted Label: {new_prediction}")

New Sample Text: 'A tomato leaf with early blight symptoms, including dark spots and yellowing.'
Predicted Label: Tomato Septoria leaf spot


In [ ]:
new_sample_text_2 = "A tomato leaf with mosaic patterns, light and dark green areas, and some leaf curling."
prediction_new_2 = predict_disease_label(new_sample_text_2)

print(f"New Sample Text: '{new_sample_text_2}'\nPredicted Label: {prediction_new_2}")

New Sample Text: 'A tomato leaf with mosaic patterns, light and dark green areas, and some leaf curling.'
Predicted Label: Tomato healthy


: 